In [1]:
import pandas as pd

In [3]:
pd.set_option('display.max_row', 1000)
df = pd.read_csv('Data\Train.csv')
df.head()

,ID,text,label
0,SUAVK39Z,I feel that it was better I dieAm happy,Depression
1,9JDAGUV3,Why do I get hallucinations?,Drugs
2,419WR1LQ,I am stresseed due to lack of financial suppor...,Depression
3,6UY7DX6Q,Why is life important?,Suicide
4,FYC0FTFB,How could I be helped to go through the depres...,Depression


In [8]:
train = df.iloc[:, :-1]
test = pd.read_csv('Data/Test.csv')
target = df.iloc[:, -1]

In [9]:
print(train.shape)
print(test.shape)
print(target.shape)

(616, 2)
(309, 2)
(616,)


In [35]:
text = pd.concat([train, test], ignore_index=True)
text.shape

(925, 2)

In [20]:
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Yaçin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [21]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [51]:
corpus = []
for i in range(0, 925):
    feeling = re.sub('[^a-zA-Z]', ' ', str(text['text'][i]))
    feeling = feeling.lower()
    feeling = feeling.split()
    ps = PorterStemmer()
    feeling = [ps.stem(word) for word in feeling if not word in set(stopwords.words('english'))]
    feeling = ' '.join(feeling)
    corpus.append(feeling)

In [57]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
result = cv.fit_transform(corpus).toarray()

In [58]:
result.shape

(925, 778)

In [69]:
dataframe = pd.DataFrame(result)
dataframe.tail()

,0,1,2,3,4,5,6,7,8,9,...,768,769,770,771,772,773,774,775,776,777
920,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
921,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
922,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
923,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
924,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [77]:
training = df.iloc[:,0].to_frame()
training = training.join(dataframe.iloc[:616,:])
training = training.join(target)
training.head()

,ID,0,1,2,3,4,5,6,7,8,...,769,770,771,772,773,774,775,776,777,label
0,SUAVK39Z,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Depression
1,9JDAGUV3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Drugs
2,419WR1LQ,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Depression
3,6UY7DX6Q,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Suicide
4,FYC0FTFB,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Depression


In [60]:
training.shape

(616, 780)

In [83]:
testing = test.iloc[:,0].to_frame()
_test = dataframe.iloc[616:,:].reset_index()
_test.drop(columns = 'index', inplace = True)
testing = testing.join(_test)
testing.head()

,ID,0,1,2,3,4,5,6,7,8,...,768,769,770,771,772,773,774,775,776,777
0,02V56KMO,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,03BMGTOK,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,03LZVFM6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0EPULUM5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0GM4C5GD,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [86]:
X = training.iloc[:,1:779].values
y = training.iloc[:,-1].values

In [87]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.15, random_state = 0)

In [100]:
from sklearn.svm import SVC
classifier = SVC(probability=True)
classifier.fit(X_train, y_train)

C:\ProgramData\Anaconda3\lib\site-packages\sklearn\svm\base.py:193: FutureWarning: The default value of gamma will change from 'auto' to 'scale' in version 0.22 to account better for unscaled features. Set gamma explicitly to 'auto' or 'scale' to avoid this warning.
  "avoid this warning.", FutureWarning)


SVC(C=1.0, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma='auto_deprecated',
    kernel='rbf', max_iter=-1, probability=True, random_state=None,
    shrinking=True, tol=0.001, verbose=False)

In [112]:
import numpy as np
y_pred = classifier.predict_proba(X_test)
np.around(y_pred, decimals=2)

array([[0.02, 0.91, 0.03, 0.04],
       [0.02, 0.93, 0.01, 0.03],
       [0.05, 0.85, 0.03, 0.06],
       [0.07, 0.81, 0.04, 0.08],
       [0.07, 0.74, 0.08, 0.11],
       [0.13, 0.07, 0.55, 0.26],
       [0.03, 0.9 , 0.02, 0.05],
       [0.08, 0.63, 0.15, 0.13],
       [0.89, 0.02, 0.02, 0.07],
       [0.02, 0.93, 0.01, 0.03],
       [0.51, 0.03, 0.32, 0.15],
       [0.04, 0.87, 0.03, 0.07],
       [0.92, 0.01, 0.02, 0.04],
       [0.01, 0.93, 0.01, 0.04],
       [0.98, 0.  , 0.01, 0.01],
       [0.95, 0.01, 0.02, 0.03],
       [0.11, 0.12, 0.55, 0.22],
       [0.12, 0.64, 0.06, 0.18],
       [0.02, 0.91, 0.03, 0.04],
       [0.02, 0.94, 0.01, 0.03],
       [0.12, 0.69, 0.07, 0.11],
       [0.46, 0.26, 0.15, 0.13],
       [0.09, 0.72, 0.07, 0.12],
       [0.1 , 0.69, 0.07, 0.14],
       [0.9 , 0.02, 0.02, 0.07],
       [0.15, 0.63, 0.07, 0.15],
       [0.95, 0.  , 0.03, 0.02],
       [0.02, 0.93, 0.01, 0.04],
       [0.03, 0.92, 0.01, 0.03],
       [0.02, 0.93, 0.01, 0.03],
       [0.

In [113]:
y_test

array(['Depression', 'Depression', 'Depression', 'Depression',
       'Depression', 'Drugs', 'Depression', 'Depression', 'Alcohol',
       'Depression', 'Alcohol', 'Depression', 'Alcohol', 'Depression',
       'Alcohol', 'Alcohol', 'Drugs', 'Depression', 'Suicide',
       'Depression', 'Depression', 'Alcohol', 'Depression', 'Depression',
       'Alcohol', 'Suicide', 'Alcohol', 'Depression', 'Depression',
       'Depression', 'Depression', 'Depression', 'Depression',
       'Depression', 'Alcohol', 'Alcohol', 'Depression', 'Depression',
       'Depression', 'Depression', 'Alcohol', 'Depression', 'Depression',
       'Alcohol', 'Alcohol', 'Depression', 'Depression', 'Depression',
       'Depression', 'Depression', 'Depression', 'Depression', 'Alcohol',
       'Drugs', 'Depression', 'Alcohol', 'Depression', 'Depression',
       'Depression', 'Depression', 'Depression', 'Depression', 'Alcohol',
       'Depression', 'Alcohol', 'Depression', 'Depression', 'Depression',
       'Suicide', 'Alc

In [125]:
Sub= testing.iloc[:,1:779].values
preds = classifier.predict_proba(Sub)
preds = np.around(preds, decimals=3)

In [126]:
Submission = testing.iloc[:, 0].to_frame()
 # Depression,Alcohol,Suicide,Drugs
Submission['Depression']=preds[:,1]
Submission['Alcohol']=preds[:,0]
Submission['Drugs']=preds[:,2]
Submission['Suicide']=preds[:,3]
Submission.head(10)

,ID,Depression,Alcohol,Drugs,Suicide
0,02V56KMO,0.827,0.057,0.037,0.079
1,03BMGTOK,0.916,0.017,0.020,0.047
2,03LZVFM6,0.990,0.002,0.003,0.005
3,0EPULUM5,0.863,0.019,0.044,0.074
4,0GM4C5GD,0.111,0.266,0.457,0.166
5,0IYM8BYT,0.711,0.099,0.094,0.096
6,0JFJVD28,0.760,0.086,0.058,0.097
7,0OZGUFOF,0.885,0.030,0.031,0.054
8,0WP999ZX,0.004,0.969,0.008,0.019
9,0ZA45Y5F,0.218,0.558,0.091,0.133


In [127]:
test.head(10)

,ID,text
0,02V56KMO,How to overcome bad feelings and emotions
1,03BMGTOK,I feel like giving up in life
2,03LZVFM6,I was so depressed feel like got no strength t...
3,0EPULUM5,I feel so low especially since I had no one to...
4,0GM4C5GD,can i be successful when I am a drug addict?
5,0IYM8BYT,"I am dissapointed,I need a sense of closure"
6,0JFJVD28,I lost hope
7,0OZGUFOF,I feel unloved and uncared for
8,0WP999ZX,What are factors that leads to addiction of al...
9,0ZA45Y5F,"Is the advice of drinking moderatly ,prudent?"


In [128]:
Submission.to_csv('Submissions/Submission_v0.csv', index=False)